# CN7 불량 요인 탐색 — 컬럼 연관성·종합 지표·24개 컬럼 근거 확인

위에서부터 순서대로 실행합니다. 원본 `moldset_labeled_cn7.csv`만 사용하며 파일을 수정하지 않습니다.

1. 선택 컬럼 시계열(인덱스 범위 먼저 제한)
2. 컬럼 간 순위상관
3. 종합 지표: PCA 평면·마할라노비스 거리·주성분 시계열
4. 24개 컬럼 근거 확인: 급변 이벤트 z-점수, 나머지 불량의 전체 정상 대비 차이
5. 같은 시기 정상과 비교(시기 효과 제거)

z-점수와 공분산은 **정상 행만** 기준으로 적합합니다. 결과 그림과 표는 `output/cn7_defect_factor_eda/`에 저장합니다.

## 0. 설정

In [ ]:
INDEX_RANGE = (0, 375)          # 시각화·상관·종합 지표 구간 [시작, 끝)
FOCUS_COLUMNS = ['Plasticizing_Position', 'Max_Injection_Speed', 'Max_Back_Pressure',
                 'Average_Back_Pressure', 'Mold_Temperature_3', 'Mold_Temperature_4']
EVENT_RANGE = (116, 120)        # 급변 이벤트 구간(양끝 포함)
EVENT_MARGIN = 2                # 이벤트 앞뒤로 제외할 여유 행 수
LOCAL_WINDOW = (80, 130)        # 같은 시기 비교 구간(양끝 포함)
MD_QUANTILE = 0.99              # 마할라노비스 기준선: 정상 거리의 분위
TOP_N = 10

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display, Image
from scipy.stats import mannwhitneyu
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/origin').is_dir())
OUT = ROOT/'output/cn7_defect_factor_eda'
OUT.mkdir(parents=True, exist_ok=True)
_names = {f.name for f in font_manager.fontManager.ttflist}
plt.rcParams['font.family'] = next((f for f in ['Malgun Gothic', 'NanumGothic', 'AppleGothic',
                                               'Noto Sans CJK KR', 'Noto Sans CJK JP'] if f in _names), 'sans-serif')
plt.rcParams['axes.unicode_minus'] = False

df = pd.read_csv(ROOT/'data/origin/moldset_labeled_cn7.csv', index_col=0, float_precision='round_trip')
FEATURES = [c for c in df.columns if c != 'PassOrFail']
assert len(FEATURES) == 24 and set(df.PassOrFail) == {0, 1}
assert all(c in FEATURES for c in FOCUS_COLUMNS)
is_defect = df.PassOrFail.eq(1)
normal = df.loc[~is_defect, FEATURES]
print(f'전체 {len(df)}행 | 불량 {int(is_defect.sum())}행 | 불량 인덱스: {list(df.index[is_defect])}')


def save_show(fig, name):
    fig.tight_layout(); fig.savefig(OUT/name, dpi=130); plt.close(fig)
    display(Image(filename=str(OUT/name)))


def zscore(rows, reference):
    std = reference.std().replace(0, np.nan)          # 상수 컬럼은 NaN
    return (rows[FEATURES] - reference.mean()) / std

## 1. 선택 컬럼 시계열

인덱스 범위를 먼저 자른 뒤 그 구간의 최솟값~최댓값으로 y축을 잡습니다. 빨간 점은 불량입니다.

In [ ]:
window = df.iloc[INDEX_RANGE[0]:INDEX_RANGE[1]]
w_bad = window.PassOrFail.eq(1)
fig, axes = plt.subplots(len(FOCUS_COLUMNS), 1, figsize=(14, 2.3*len(FOCUS_COLUMNS)), sharex=True)
for ax, col in zip(axes, FOCUS_COLUMNS):
    ax.plot(window.index, window[col], lw=1, color='#4C78A8')
    ax.scatter(window.index[w_bad], window.loc[w_bad, col], color='#E45756', s=25, zorder=3, label='불량')
    lo, hi = window[col].min(), window[col].max(); pad = (hi-lo)*0.08 or 0.1
    ax.set_ylim(lo-pad, hi+pad); ax.set_ylabel(col, fontsize=9); ax.grid(alpha=.3)
axes[0].legend(loc='upper right'); axes[-1].set_xlabel('인덱스')
axes[-1].set_xlim(window.index.min(), window.index.max())
fig.suptitle(f'CN7 labeled — 인덱스 {INDEX_RANGE[0]}:{INDEX_RANGE[1]}', y=1.0)
save_show(fig, '01_focus_timeseries.png')
display(window[FOCUS_COLUMNS].agg(['min', 'max', 'nunique']).T.round(4))

## 2. 컬럼 간 순위상관(Spearman)

In [ ]:
corr = window[FOCUS_COLUMNS].corr(method='spearman')
corr.to_csv(OUT/'02_focus_spearman.csv')
fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha='right'); ax.set_yticks(range(len(corr)), corr.index)
for i in range(len(corr)):
    for j in range(len(corr)):
        v = corr.iloc[i, j]; ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=9,
                                     color='white' if abs(v) > .6 else 'black')
fig.colorbar(im, ax=ax, fraction=.046); ax.set_title('컬럼 간 순위상관')
save_show(fig, '02_focus_spearman.png')

## 3. 종합 지표

표준화·PCA·공분산은 구간 안의 **정상 행**으로만 적합합니다. 마할라노비스 거리는 컬럼 간 상관을 반영한 "정상에서 떨어진 정도"입니다.

In [ ]:
Xw = window[FOCUS_COLUMNS]
scaler = StandardScaler().fit(Xw[~w_bad])
Z = pd.DataFrame(scaler.transform(Xw), index=window.index, columns=FOCUS_COLUMNS)
pca = PCA(n_components=2).fit(Z[~w_bad]); pc = pca.transform(Z)
loadings = pd.DataFrame(pca.components_, index=['PC1', 'PC2'], columns=FOCUS_COLUMNS)
display(loadings.round(2)); loadings.to_csv(OUT/'03_pca_loadings.csv')

cov_inv = np.linalg.pinv(np.cov(Z[~w_bad].T)); mu = Z[~w_bad].mean().to_numpy()
diff = Z.to_numpy() - mu
md_dist = np.sqrt(np.einsum('ij,jk,ik->i', diff, cov_inv, diff))
md_thr = float(np.quantile(md_dist[~w_bad.to_numpy()], MD_QUANTILE))
composite = pd.DataFrame({'PassOrFail': window.PassOrFail, 'PC1': pc[:, 0], 'PC2': pc[:, 1],
                          'mahalanobis': md_dist, 'over_threshold': md_dist > md_thr}, index=window.index)
composite.to_csv(OUT/'03_composite_scores.csv')
print(f'기준선(정상 {MD_QUANTILE:.0%} 분위) = {md_thr:.2f}')
print('기준선 초과 — 불량:', int(composite.loc[w_bad, 'over_threshold'].sum()), '/', int(w_bad.sum()),
      '| 정상:', int(composite.loc[~w_bad, 'over_threshold'].sum()), '/', int((~w_bad).sum()))

fig = plt.figure(figsize=(15, 10)); gs = fig.add_gridspec(3, 1)
ax = fig.add_subplot(gs[0]); b = w_bad.to_numpy()
ax.scatter(pc[~b, 0], pc[~b, 1], s=12, alpha=.5, label='정상')
ax.scatter(pc[b, 0], pc[b, 1], s=40, color='#E45756', edgecolors='k', label='불량')
ax.set(xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.0%})', ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.0%})',
       title='PCA 평면'); ax.legend()
ax = fig.add_subplot(gs[1]); ax.plot(window.index, md_dist, lw=1)
ax.scatter(window.index[b], md_dist[b], color='#E45756', s=25, zorder=3, label='불량')
ax.axhline(md_thr, color='gray', ls='--', label=f'정상 {MD_QUANTILE:.0%} 분위'); ax.set_yscale('log')
ax.set(ylabel='마할라노비스 거리(log)', title='종합 이상도'); ax.legend(); ax.grid(alpha=.3)
ax = fig.add_subplot(gs[2]); ax.plot(window.index, pc[:, 0], lw=1, label='PC1'); ax.plot(window.index, pc[:, 1], lw=1, label='PC2')
ax.scatter(window.index[b], pc[b, 0], color='#E45756', s=20, zorder=3, label='불량(PC1)')
ax.set(xlabel='인덱스', ylabel='주성분 점수', title='주성분 시계열'); ax.legend(); ax.grid(alpha=.3)
save_show(fig, '03_composite_indices.png')

## 4. 24개 컬럼 근거 확인

### 4.1 급변 이벤트

이벤트 구간의 행이 전체 정상 대비 어느 컬럼에서 크게 벗어났는지 |z| 평균으로 순위를 매깁니다. 구간 안의 고유 입력 패턴 수도 확인합니다(같은 패턴 반복이면 사실상 한 번의 사건).

In [ ]:
event = df.loc[EVENT_RANGE[0]:EVENT_RANGE[1]]
event_z = zscore(event, normal)
event_rank = pd.DataFrame({'mean_abs_z': event_z.abs().mean(), 'mean_signed_z': event_z.mean()}) \
    .sort_values('mean_abs_z', ascending=False)
event_rank.to_csv(OUT/'04_event_zscore.csv')
print(f'이벤트 {EVENT_RANGE}: {len(event)}행, 불량 {int(event.PassOrFail.sum())}행, '
      f'고유 입력 패턴 {event[FEATURES].drop_duplicates().shape[0]}개')
display(event_rank.head(TOP_N).round(2))

### 4.2 나머지 불량 vs 전체 정상

이벤트(앞뒤 여유 포함)를 제외한 불량을 전체 정상과 비교합니다. 연속된 행이라 독립이 아니므로 p값은 순위 참고용입니다.

In [ ]:
event_idx = range(EVENT_RANGE[0]-EVENT_MARGIN, EVENT_RANGE[1]+EVENT_MARGIN+1)
others = df.loc[is_defect & ~df.index.isin(event_idx)]


def compare(defects, reference, label):
    z = zscore(defects, reference)
    rows = []
    for col in FEATURES:
        if reference[col].nunique() < 2 and defects[col].nunique() < 2:
            continue
        p = mannwhitneyu(defects[col], reference[col], alternative='two-sided').pvalue
        rows.append(dict(column=col, mean_signed_z=z[col].mean(), mean_abs_z=z[col].abs().mean(), p_value=p))
    table = pd.DataFrame(rows).sort_values('p_value').reset_index(drop=True)
    table.insert(0, 'comparison', label)
    return table

global_cmp = compare(others, normal, '전체 정상')
print(f'나머지 불량 {len(others)}행: {list(others.index)}')
display(global_cmp.head(TOP_N).round(4))

## 5. 같은 시기 정상과 비교

나머지 불량이 특정 시기에 몰려 있으면, 4.2의 차이가 불량 때문인지 그 시기의 운전 조건 때문인지 구분되지 않습니다. `LOCAL_WINDOW` 안의 정상만 기준으로 다시 비교해 두 결과가 같은 컬럼을 가리키는지 확인합니다.

In [ ]:
local_normal = df.loc[LOCAL_WINDOW[0]:LOCAL_WINDOW[1]]
local_normal = local_normal.loc[local_normal.PassOrFail.eq(0), FEATURES]
local_defects = others.loc[LOCAL_WINDOW[0]:LOCAL_WINDOW[1]]
local_cmp = compare(local_defects, local_normal, f'같은 시기 정상 {LOCAL_WINDOW}')
print(f'비교: 불량 {len(local_defects)}행 vs 같은 시기 정상 {len(local_normal)}행')
display(local_cmp.head(TOP_N).round(4))

both = global_cmp.set_index('column')[['mean_signed_z', 'p_value']].join(
    local_cmp.set_index('column')[['mean_signed_z', 'p_value']], lsuffix='_전체', rsuffix='_같은시기')
both.to_csv(OUT/'05_global_vs_local.csv')

top = both.sort_values('p_value_전체').head(TOP_N)
fig, ax = plt.subplots(figsize=(10, 5.5)); ypos = np.arange(len(top))
ax.barh(ypos-0.2, top['mean_signed_z_전체'], height=0.4, label='전체 정상 기준')
ax.barh(ypos+0.2, top['mean_signed_z_같은시기'], height=0.4, label='같은 시기 정상 기준')
ax.set_yticks(ypos, top.index); ax.invert_yaxis(); ax.axvline(0, color='k', lw=.8)
ax.set(xlabel='불량의 평균 z (기준 정상 대비)', title='나머지 불량: 기준을 바꾸면 차이가 유지되는가'); ax.legend()
save_show(fig, '05_global_vs_local.png')

## 6. 요약

아래 문장은 위 결과에서 자동으로 채웁니다. 해석은 가설이며 공정 가이드북과 대조가 필요합니다.

In [ ]:
kept = both.loc[(both.p_value_전체 < 0.01) & (both.p_value_같은시기 < 0.05)].index.tolist()
lost = both.loc[(both.p_value_전체 < 0.01) & (both.p_value_같은시기 >= 0.05)].index.tolist()
summary = f"""# CN7 불량 요인 탐색 요약

- 급변 이벤트 {EVENT_RANGE}: 상위 컬럼 {', '.join(event_rank.index[:5])} (|z| {', '.join(f'{v:.1f}' for v in event_rank.mean_abs_z[:5])}); 고유 패턴 {event[FEATURES].drop_duplicates().shape[0]}개
- 선택 컬럼 마할라노비스 기준선 초과 불량: {int(composite.loc[w_bad, 'over_threshold'].sum())}/{int(w_bad.sum())}
- 나머지 불량 {len(others)}행 — 전체 정상 대비 p<0.01 컬럼 중
  - 같은 시기 비교에서도 유지(p<0.05): {', '.join(kept) or '없음'}
  - 같은 시기 비교에서 사라짐(시기 효과 의심): {', '.join(lost) or '없음'}
"""
(OUT/'analysis_summary.md').write_text(summary, encoding='utf-8'); print(summary)